In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
tourism_path =r"C:\Users\hyunj\Seoul_Strolling_Adventure\크롤링\전국.xlsx"
df_tourism = pd.read_excel(tourism_path)

df_tourism.head(2)

,addr1,addr2,areacode,booktour,cat1,cat2,cat3,contentid,contenttypeid,createdtime,...,firstimage2,cpyrhtDivCd,mapx,mapy,mlevel,modifiedtime,sigungucode,tel,title,zipcode
0,충청남도 공주시 감영길 3,(반죽동),34.0,NaN,쇼핑,쇼핑,전문매장/상가,2750144,38,20210928012320,...,NaN,NaN,127.121672,36.452930,6.0,20241226163730,1.0,NaN,가가상점,32546
1,부산광역시 부산진구 중앙번영로 (6),NaN,6.0,NaN,음식,음식점,일식,2805408,39,20220125140006,...,NaN,NaN,129.059828,35.144807,6.0,20240104133528,7.0,NaN,가가와,47361


In [4]:
transport_path=r"C:\Users\hyunj\Seoul_Strolling_Adventure\dataset\대중교통위치\관광지_대중교통_매핑결과.xlsx"
df_transport = pd.read_excel(transport_path)

df_transport.head(2)

,addr1,cat1,cat2,cat3,mapx,mapy,title,closest_subway_station,closest_subway_line,closest_bus_station,정류장명,정류장번호
0,충청남도 공주시 감영길 3,쇼핑,쇼핑,전문매장/상가,127.121672,36.45293,가가상점,반석역,['대전 도시철도 1호선'],사대부고,사대부고,CCB250002217
1,충청남도 공주시 감영길 3,쇼핑,쇼핑,전문매장/상가,127.121672,36.45293,가가상점,반석역,['대전 도시철도 1호선'],사대부고,사대부고,JEB405002114


In [ ]:
df = pd.merge(df_tourism, df_transport, on=['title', 'addr1'], how='inner')

In [6]:
df.head(2)

,addr1,addr2,areacode,booktour,cat1_x,cat2_x,cat3_x,contentid,contenttypeid,createdtime,...,cat1_y,cat2_y,cat3_y,mapx_y,mapy_y,closest_subway_station,closest_subway_line,closest_bus_station,정류장명,정류장번호
0,충청남도 공주시 감영길 3,(반죽동),34.0,NaN,쇼핑,쇼핑,전문매장/상가,2750144,38,20210928012320,...,쇼핑,쇼핑,전문매장/상가,127.121672,36.45293,반석역,['대전 도시철도 1호선'],사대부고,사대부고,CCB250002217
1,충청남도 공주시 감영길 3,(반죽동),34.0,NaN,쇼핑,쇼핑,전문매장/상가,2750144,38,20210928012320,...,쇼핑,쇼핑,전문매장/상가,127.121672,36.45293,반석역,['대전 도시철도 1호선'],사대부고,사대부고,JEB405002114


In [13]:
# 중복 접미사 정리(존재할 때만)
df = df.rename(columns={
    'cat1_x': 'cat1', 'cat2_x': 'cat2', 'cat3_x': 'cat3',
    'mapx_x': 'mapx', 'mapy_x': 'mapy'
})
df.drop(columns=[c for c in ['cat1_y','cat2_y','cat3_y','mapx_y','mapy_y'] if c in df.columns],
        inplace=True, errors='ignore')

In [14]:
# 접근성 점수
df['access_score'] = df['closest_subway_station'].notna().astype(int) + df['closest_bus_station'].notna().astype(int)

In [15]:
df.head(2)

,addr1,addr2,areacode,booktour,cat1,cat2,cat3,contentid,contenttypeid,createdtime,...,zipcode,closest_subway_station,closest_subway_line,closest_bus_station,정류장명,정류장번호,cat1_score,modified_score,density_score,access_score
0,충청남도 공주시 감영길 3,(반죽동),34.0,NaN,쇼핑,쇼핑,전문매장/상가,2750144,38,20210928012320,...,32546,반석역,['대전 도시철도 1호선'],사대부고,사대부고,CCB250002217,4.364726,9.662452,0.118101,2
1,충청남도 공주시 감영길 3,(반죽동),34.0,NaN,쇼핑,쇼핑,전문매장/상가,2750144,38,20210928012320,...,32546,반석역,['대전 도시철도 1호선'],사대부고,사대부고,JEB405002114,4.364726,9.662452,0.118101,2


In [16]:
# 2. 카테고리 점수 (단순 개수 기준)
cat1_freq = df['cat1'].value_counts()
df['cat1_score'] = df['cat1'].map(lambda x: cat1_freq.get(x, 0))
df['cat1_score'] = 10 * (df['cat1_score'] - df['cat1_score'].min()) / (df['cat1_score'].max() - df['cat1_score'].min())


In [17]:
# 3. 날짜 점수: 최근일수록 높게
df['modifiedtime'] = pd.to_datetime(df['modifiedtime'], errors='coerce', format='%Y%m%d%H%M%S')
df['modified_score'] = 10 * (df['modifiedtime'] - df['modifiedtime'].min()) / (df['modifiedtime'].max() - df['modifiedtime'].min())


In [18]:
# 4. 밀집도 점수
from sklearn.neighbors import NearestNeighbors
coords = df[['mapy', 'mapx']].dropna().values
nn = NearestNeighbors(n_neighbors=6).fit(coords)
distances, _ = nn.kneighbors(coords)
density_score = 1 / (distances[:, 1:].mean(axis=1) + 1e-6)
df.loc[df[['mapy', 'mapx']].dropna().index, 'density_score'] = 10 * (density_score - density_score.min()) / (density_score.max() - density_score.min())


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.base import clone

In [23]:
# --- 전제: df 안에 아래 5개 컬럼이 이미 계산되어 있음 ---
Y_COLS = ['access_score', 'cat1_score', 'modified_score', 'density_score']
X_COL = 'title'

In [24]:
need_cols = [X_COL] + Y_COLS
df_train = df.dropna(subset=need_cols).copy()
X_all = df_train[X_COL].astype(str).to_numpy()
Y_all = df_train[Y_COLS].to_numpy()  # (n, 4)

In [25]:
# 후보 파이프라인: TF-IDF → SVD(밀집화) → MultiOutput(모델)
def make_pipe(estimator, n_comp=64):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            analyzer='char', ngram_range=(2,4),
            min_df=1, max_features=20000
        )),
        ('svd', TruncatedSVD(n_components=n_comp, random_state=0)),
        ('mo', MultiOutputRegressor(estimator))
    ])

base_pipes = {
    "Ridge": make_pipe(Ridge(alpha=2.0, random_state=0)),
    "GBDT":  make_pipe(GradientBoostingRegressor(
                n_estimators=400, learning_rate=0.05, max_depth=3,
                subsample=0.9, random_state=0)),
    "HGBR":  make_pipe(HistGradientBoostingRegressor(
                max_depth=None, max_iter=400, learning_rate=0.05,
                random_state=0))
}


In [ ]:
# 반복 홀드아웃(30회): 타깃별 R² 평균 → 평균
seeds = range(30)
avg_scores = {}
for name, pipe in base_pipes.items():
    r2_list = []
    for seed in seeds:
        p = clone(pipe)
        X_tr, X_te, Y_tr, Y_te = train_test_split(X_all, Y_all, test_size=0.2, random_state=seed)
        p.fit(X_tr, Y_tr)
        Y_pred = p.predict(X_te)
        r2_targets = [r2_score(Y_te[:, i], Y_pred[:, i]) for i in range(Y_te.shape[1])]
        r2_list.append(np.mean(r2_targets))
    avg_scores[name] = float(np.mean(r2_list))

print("모델별 평균 R²(타깃 평균, 30회):")
for k, v in sorted(avg_scores.items(), key=lambda x: -x[1]):
    print(f"  {k:5s}: {v:.4f}")

best_model_name = max(avg_scores, key=avg_scores.get)
print(f"\n✅ 최종 선택된 모델: {best_model_name}")

In [ ]:
# 전체 학습 후 예측
best_pipe = clone(base_pipes[best_model_name])
best_pipe.fit(X_all, Y_all)
Y_hat = best_pipe.predict(X_all)  # (n, 4)
pred_df = pd.DataFrame(Y_hat, columns=[f'pred_{c}' for c in Y_COLS], index=df_train.index)

In [ ]:
# =========================
# 3) model_score + tour_score
# =========================
# access_score 0~2 → [0,2] clip 후 ×5 해서 0~10 정렬
pred_access_10   = np.clip(pred_df['pred_access_score'].to_numpy(), 0.0, 2.0) * 5.0
pred_cat1_10     = np.clip(pred_df['pred_cat1_score'].to_numpy(),     0.0, 10.0)
pred_modified_10 = np.clip(pred_df['pred_modified_score'].to_numpy(), 0.0, 10.0)
pred_density_10  = np.clip(pred_df['pred_density_score'].to_numpy(),  0.0, 10.0)

df.loc[df_train.index, 'model_score'] = (pred_access_10 + pred_cat1_10 + pred_modified_10 + pred_density_10) / 4.0

In [ ]:
# 최종 관광지수 (가중치 균등)
df.loc[df_train.index, 'tour_score'] = (
    df.loc[df_train.index, 'access_score'] +
    df.loc[df_train.index, 'cat1_score'] +
    df.loc[df_train.index, 'density_score'] +
    df.loc[df_train.index, 'modified_score'] +
    df.loc[df_train.index, 'model_score']
) / 5.0

In [ ]:

# =========================
# 4) 결과 확인/저장
# =========================
out_cols = ['title','addr1','tour_score','model_score'] + Y_COLS
df_sorted = df.loc[df_train.index, out_cols].sort_values('tour_score', ascending=False).reset_index(drop=True)

print("\n상위 10개:")
print(df_sorted.head(10))

In [ ]:
output_path = r"C:\Users\hyunj\Seoul_Strolling_Adventure\dataset\지역\관광지수_매핑결과.csv"

# NaN 또는 빈 문자열("")만 있는 열 제거
df_sorted = df_sorted.dropna(axis=1, how='all')  # 모든 값이 NaN인 열 제거
df_sorted = df_sorted.loc[:, ~(df_sorted == '').all()]  # 모든 값이 빈 문자열("")인 열 제거

# 엑셀로 저장
df_sorted.to_csv(output_path, index=False)
print(f"✅ 결과가 '{output_path}'에 저장되었습니다.")

✅ 결과가 'C:\Users\hyunj\Seoul_Strolling_Adventure\dataset\지역\관광지수_매핑결과.csv'에 저장되었습니다.
